In [1]:
import joblib
import pandas as pd
import numpy as np

In [2]:
# unused dataset 
data = pd.read_csv('ds/fraudTest.csv' , nrows=2000)

data.tail(5)

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
1995,1995,2020-06-21 23:49:25,345933964507467,fraud_O'Hara-Wilderman,food_dining,2.64,Carol,Dillon,F,27479 Reeves Dale,...,38.4121,-75.2811,718,Regulatory affairs officer,1985-03-19,db9609979b2c8c9af5c60a02448895ac,1371858565,38.579881,-74.493330,0
1996,1996,2020-06-21 23:49:36,4607072969078276,"fraud_Streich, Rolfson and Wilderman",kids_pets,86.23,Brenda,Perez,F,033 Tara Brook Suite 523,...,35.8985,-97.2607,1493,Amenity horticulturist,1985-03-21,a58c8deeb72c9ee7646c741a58ce7c26,1371858576,35.069578,-97.572451,0
1997,1997,2020-06-21 23:49:49,6011504998544485,"fraud_Boehm, Predovic and Reinger",misc_pos,36.13,Ashley,Whitney,F,4038 Smith Avenue,...,32.5104,-86.8138,1089,Materials engineer,1971-11-02,9fbee0d9611b6a85b824bdfd1b3acd13,1371858589,33.010868,-86.798841,0
1998,1998,2020-06-21 23:49:58,374656033243756,fraud_Hickle Group,shopping_pos,3.32,David,Lewis,M,1499 Michael Rue,...,38.8954,-77.1633,207410,Mudlogger,1984-07-03,ac50f5323801477384fc7a129736c99c,1371858598,38.787164,-77.216642,0
1999,1999,2020-06-21 23:50:33,38530489946071,fraud_Bernhard-Lesch,food_dining,7.08,Laura,Johns,F,95835 Garcia Rue,...,34.9572,-81.9916,530,Animal technologist,1989-05-14,01e950b58e869d295e490e2ccda89d22,1371858633,34.331410,-82.671540,0


In [ ]:
#test model parameters
try:
    model = joblib.load('parameters/rf_fraud_model.pkl')
    scaler = joblib.load('parameters/scaler.pkl')
    le_cat = joblib.load('parameters/categorie_encoder.pkl')
    le_gen = joblib.load('parameters/gender_encoder.pkl')
    le_job = joblib.load('parameters/job_encoder.pkl')
    print("success Load")
except FileNotFoundError as e:
    print(f"file unavailable {e}")
    exit()


print("_____Transaction Fraud Detector")
print("____enter same info in dataset ---")
amt = float(input(" Transaction Amount: "))
category = input(" Category (e.g., shopping_net, food_dining): ")
gender = input(" Gender (M/F): ")
city_pop = int(input(" City Population: "))
job = input(" Job Title : ")
lat = float(input(" User Latitude: "))
long = float(input(" User Longitude: "))
merch_lat = float(input(" Merchant Latitude: "))
merch_long = float(input(" Merchant Longitude: "))
age = int(input(" User Age: "))


hour = int(input("Enter Hour (0-23): "))
day = int(input("Enter Day (1-31): "))
month = int(input("Enter Month (1-12): "))
year = int(input("Enter Year: "))

#use same encoder and scaller
data_dict = {
    'category': [category],
    'amt': [amt],
    'gender': [gender],
    'lat': [lat],
    'long': [long],
    'city_pop': [city_pop],
    'job': [job],
    'merch_lat': [merch_lat],
    'merch_long': [merch_long],
    'trans_year': [year],
    'trans_month': [month],
    'trans_day': [day],
    'trans_hour': [hour],
    'age': [age]
}

df_input = pd.DataFrame(data_dict)


try:
    
    df_input['category'] = le_cat.transform(df_input['category'])
    df_input['gender'] = le_gen.transform(df_input['gender'])
    df_input['job'] = le_job.transform(df_input['job'])

    # Scaling
    to_scale = ['amt', 'lat', 'long', 'city_pop', 'merch_lat', 'merch_long', 
                'trans_year', 'trans_hour', 'trans_day', 'trans_month', 'age']
    df_input[to_scale] = scaler.transform(df_input[to_scale])

   
    prediction = model.predict(df_input)
    #model prediction
    probability = model.predict_proba(df_input)[0][1] 

    print("\n" + "="*30)
    if prediction[0] == 1:
        print(f"FRAUD DETECTED! class 1")
        print(f"probability : {probability * 100:.2f}%")
    else:
        print(f"Ok class 0")
        print(f"probability : {(1 - probability) * 100:.2f}%")
    print("="*30)

except ValueError as e:
    print(f"\n error")
    print(f"why : {e}")